In [12]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

# Find Pair-Divergence Pairs

Calculate stock returns and construct a universe of candidate equity pairs for pair-divergence analysis.

In [2]:
# Locate repository root
currentDirectory = Path.cwd()

possibleRoots = [
    currentDirectory,
    currentDirectory.parent,
    currentDirectory.parent.parent
]

repositoryRoot = None

for directory in possibleRoots:
    marketDataPath = directory/"data"/"pairDivergence"/"pairDivergenceData.parquet"

    if marketDataPath.exists():
        repositoryRoot = directory
        break

if repositoryRoot is None:
    raise FileNotFoundError("Could not find pairDivergenceData.parquet")

marketDataPath = repositoryRoot/"data"/"pairDivergence"/"pairDivergenceData.parquet"
marketData = pd.read_parquet(marketDataPath)

marketData = marketData.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Rows: {len(marketData):,}")
print(f"Symbols: {marketData['symbol'].nunique():,}")
print(f"First date: {marketData['date'].min().date()}")
print(f"Last date: {marketData['date'].max().date()}")
print(marketData.columns.tolist())

Rows: 8,808,211
Symbols: 5,221
First date: 2018-01-02
Last date: 2026-06-30
['date', 'symbol', 'open', 'high', 'low', 'close', 'adjClose', 'volume']


In [3]:
# Calculate one-day adjusted-close log returns
tradingDates = pd.Index(sorted(marketData["date"].unique()))
tradingDayPosition = pd.Series(np.arange(len(tradingDates)), index=tradingDates)

marketData["tradingDay"] = marketData["date"].map(tradingDayPosition)
marketData["previousTradingDay"] = marketData.groupby("symbol")["tradingDay"].shift(1)
marketData["previousAdjClose"] = marketData.groupby("symbol")["adjClose"].shift(1)

consecutiveObservation = marketData["tradingDay"] - marketData["previousTradingDay"] == 1

marketData["return1"] = np.where(
    consecutiveObservation,
    np.log(marketData["adjClose"] / marketData["previousAdjClose"]),
    np.nan
)

print(f"Return observations: {marketData['return1'].notna().sum():,}")
print(f"Missing returns: {marketData['return1'].isna().sum():,}")
print(f"Finite returns: {np.isfinite(marketData['return1'].dropna()).all()}")
print(f"Nonconsecutive observations excluded: {(marketData['previousTradingDay'].notna() & ~consecutiveObservation).sum():,}")

Return observations: 8,802,980
Missing returns: 5,231
Finite returns: True
Nonconsecutive observations excluded: 10


In [4]:
# Remove temporary return-calculation columns
marketData = marketData.drop(columns=["tradingDay", "previousTradingDay", "previousAdjClose"])

## Prepare Return Matrix

Organize one-day log returns by trading date and symbol for pair screening. Missing returns are retained where a stock lacks a valid consecutive trading-day observation.

In [5]:
# Organize returns by date and symbol
returns = marketData.pivot(
    index="date",
    columns="symbol",
    values="return1"
).sort_index()

returns.columns.name = None

returnCounts = returns.notna().sum()

print(f"Trading dates: {len(returns):,}")
print(f"Symbols: {len(returns.columns):,}")
print(f"Minimum return observations: {returnCounts.min():,}")
print(f"Median return observations: {int(returnCounts.median()):,}")
print(f"Maximum return observations: {returnCounts.max():,}")
print()
print(f"Symbols with at least 252 returns: {(returnCounts >= 252).sum():,}")
print(f"Symbols with at least 504 returns: {(returnCounts >= 504).sum():,}")
print(f"Symbols with at least 756 returns: {(returnCounts >= 756).sum():,}")
print(f"Symbols with at least 1,260 returns: {(returnCounts >= 1260).sum():,}")

Trading dates: 2,134
Symbols: 5,221
Minimum return observations: 2
Median return observations: 2,133
Maximum return observations: 2,133

Symbols with at least 252 returns: 4,920
Symbols with at least 504 returns: 4,639
Symbols with at least 756 returns: 4,459
Symbols with at least 1,260 returns: 3,984


## Screen Pair Eligibility

Pairs are screened using aligned daily log returns. A pair is eligible on a given date when:

- the absolute 42-day correlation is at least 0.75;
- the absolute 126-day correlation is at least 0.50;
- the 42-day and 126-day correlations have the same sign; and
- a 252-observation correlation can be calculated.

Before evaluating pair-level correlations, stocks and stock pairs that cannot support the required return history are removed from consideration.

In [6]:
# Define pair-eligibility parameters
shortCorrelationWindow = 42
mediumCorrelationWindow = 126
longCorrelationWindow = 252

shortCorrelationThreshold = 0.75
mediumCorrelationThreshold = 0.50

In [7]:
# Screen stocks by available return history
eligibleSymbols = returnCounts[returnCounts >= longCorrelationWindow].index

excludedSymbols = returnCounts[returnCounts < longCorrelationWindow].index

possiblePairs = len(eligibleSymbols) * (len(eligibleSymbols) - 1) // 2

print(f"Eligible symbols: {len(eligibleSymbols):,}")
print(f"Excluded symbols: {len(excludedSymbols):,}")
print(f"Possible pairs after history screen: {possiblePairs:,}")

Eligible symbols: 4,920
Excluded symbols: 301
Possible pairs after history screen: 12,100,740


In [14]:
def rollingCorrelation(firstReturns, secondReturns, window):
    """
    Calculate rolling Pearson correlation for two aligned return arrays. The calculation uses
    cumulative sums to avoid repeatedly evaluating each rolling window from scratch.

    Inputs:
        - firstReturns (array): Returns for the first stock.
        - secondReturns (array): Returns for the second stock.
        - window (int): Number of observations in each rolling correlation window.

    Returns:
        - correlation (array): Rolling correlations for each completed window.
    """
    # A complete window is required before correlation can be calculated
    if len(firstReturns) < window:
        return np.array([])

    # Build cumulative sums for the correlation components
    cumulativeFirst = np.concatenate(([0.0], np.cumsum(firstReturns)))
    cumulativeSecond = np.concatenate(([0.0], np.cumsum(secondReturns)))
    cumulativeFirstSquared = np.concatenate(([0.0], np.cumsum(firstReturns ** 2)))
    cumulativeSecondSquared = np.concatenate(([0.0], np.cumsum(secondReturns ** 2)))
    cumulativeProduct = np.concatenate(([0.0], np.cumsum(firstReturns * secondReturns)))

    # Recover each rolling-window sum from the cumulative arrays
    sumFirst = cumulativeFirst[window:] - cumulativeFirst[:-window]
    sumSecond = cumulativeSecond[window:] - cumulativeSecond[:-window]
    sumFirstSquared = cumulativeFirstSquared[window:] - cumulativeFirstSquared[:-window]
    sumSecondSquared = cumulativeSecondSquared[window:] - cumulativeSecondSquared[:-window]
    sumProduct = cumulativeProduct[window:] - cumulativeProduct[:-window]

    # Calculate the unscaled covariance and variance terms
    covariance = window * sumProduct - sumFirst * sumSecond
    varianceFirst = window * sumFirstSquared - sumFirst ** 2
    varianceSecond = window * sumSecondSquared - sumSecond ** 2

    correlation = np.full(len(covariance), np.nan)

    # Correlation is undefined when either return series has zero variance
    validVariance = (varianceFirst > 0) & (varianceSecond > 0)

    correlation[validVariance] = covariance[validVariance] / np.sqrt(varianceFirst[validVariance] * varianceSecond[validVariance])

    return correlation

In [15]:
def pairEverEligible(firstReturns, secondReturns):
    """
    Determine whether a stock pair satisfies the pair-eligibility criteria on at least one
    date.

    Returns are first restricted to dates where both stocks have valid observations. Rolling
    correlations are then calculated over the aligned return history and compared using the
    defined 42, 126, and 252 day windows.

    Inputs:
        - firstReturns (array): Daily returns for the first stock.
        - secondReturns (array): Daily returns for the second stock.
    Returns:
        - bool: True if the pair is eligible at least once, otherwise False.
    """

    # Keep only dates where both stocks have valid returns
    alignedMask = np.isfinite(firstReturns) & np.isfinite(secondReturns)

    # A pair cannot qualify without enough aligned observations
    if alignedMask.sum() < longCorrelationWindow:
        return False

    alignedFirst = firstReturns[alignedMask]
    alignedSecond = secondReturns[alignedMask]

    # Calculate rolling correlations over each eligibility window
    correlation42 = rollingCorrelation(
        alignedFirst,
        alignedSecond,
        shortCorrelationWindow
    )

    correlation126 = rollingCorrelation(
        alignedFirst,
        alignedSecond,
        mediumCorrelationWindow
    )

    correlation252 = rollingCorrelation(
        alignedFirst,
        alignedSecond,
        longCorrelationWindow
    )

    # Align all correlation arrays to observations where corr252 exists
    correlation42 = correlation42[longCorrelationWindow - shortCorrelationWindow:]

    correlation126 = correlation126[longCorrelationWindow - mediumCorrelationWindow:]

    # Apply the original pair-eligibility criteria
    eligible = (
        (np.abs(correlation42) >= shortCorrelationThreshold) &
        (np.abs(correlation126) >= mediumCorrelationThreshold) &
        (np.sign(correlation42) == np.sign(correlation126)) &
        np.isfinite(correlation252)
    )

    return eligible.any()

In [19]:
# Screen the complete pair universe
eligibleReturnMatrix = returns[eligibleSymbols].to_numpy()

eligiblePairs = []

for pairNumber, (firstIndex, secondIndex) in enumerate(
    combinations(range(len(eligibleSymbols)), 2),
    start=1
):
    if pairEverEligible(
        eligibleReturnMatrix[:, firstIndex],
        eligibleReturnMatrix[:, secondIndex]
    ):
        eligiblePairs.append((
            eligibleSymbols[firstIndex],
            eligibleSymbols[secondIndex]
        ))

    if pairNumber % 500000 == 0 or pairNumber == possiblePairs:
        print(f"Processed {pairNumber:,} / {possiblePairs:,} pairs ({pairNumber / possiblePairs:.1%})")
        print(f"Eligible pairs: {len(eligiblePairs):,}")
        print()

print(f"Pairs screened: {possiblePairs:,}")
print(f"Eligible pairs: {len(eligiblePairs):,}")

Processed 500,000 / 12,100,740 pairs (4.1% )
Eligible pairs: 78,367

Processed 1,000,000 / 12,100,740 pairs (8.3% )
Eligible pairs: 153,285

Processed 1,500,000 / 12,100,740 pairs (12.4% )
Eligible pairs: 227,995

Processed 2,000,000 / 12,100,740 pairs (16.5% )
Eligible pairs: 294,756

Processed 2,500,000 / 12,100,740 pairs (20.7% )
Eligible pairs: 387,751

Processed 3,000,000 / 12,100,740 pairs (24.8% )
Eligible pairs: 464,348

Processed 3,500,000 / 12,100,740 pairs (28.9% )
Eligible pairs: 542,710

Processed 4,000,000 / 12,100,740 pairs (33.1% )
Eligible pairs: 629,231

Processed 4,500,000 / 12,100,740 pairs (37.2% )
Eligible pairs: 713,771

Processed 5,000,000 / 12,100,740 pairs (41.3% )
Eligible pairs: 790,850

Processed 5,500,000 / 12,100,740 pairs (45.5% )
Eligible pairs: 879,264

Processed 6,000,000 / 12,100,740 pairs (49.6% )
Eligible pairs: 949,171

Processed 6,500,000 / 12,100,740 pairs (53.7% )
Eligible pairs: 1,035,656

Processed 7,000,000 / 12,100,740 pairs (57.8% )
Eligib

In [20]:
# Construct the eligible pair universe
pairUniverse = pd.DataFrame(
    eligiblePairs,
    columns=["symbolA", "symbolB"]
)

print(f"Pairs: {len(pairUniverse):,}")
print(f"Unique symbols: {pd.unique(pairUniverse[['symbolA', 'symbolB']].values.ravel()).size:,}")
print(pairUniverse.head())

Pairs: 1,914,434
Unique symbols: 4,679
  symbolA symbolB
0       A      AA
1       A     AAL
2       A    AAMI
3       A    AAOI
4       A    AAON


In [21]:
# Validate the eligible pair universe
duplicatePairs = pairUniverse.duplicated().sum()
sameSymbolPairs = (pairUniverse["symbolA"] == pairUniverse["symbolB"]).sum()
unorderedPairs = (pairUniverse["symbolA"] >= pairUniverse["symbolB"]).sum()

pairSymbols = pd.unique(pairUniverse[["symbolA", "symbolB"]].values.ravel())
invalidSymbols = ~np.isin(pairSymbols, eligibleSymbols)

print(f"Duplicate pairs: {duplicatePairs:,}")
print(f"Same-symbol pairs: {sameSymbolPairs:,}")
print(f"Incorrectly ordered pairs: {unorderedPairs:,}")
print(f"Symbols outside eligible universe: {invalidSymbols.sum():,}")

Duplicate pairs: 0
Same-symbol pairs: 0
Incorrectly ordered pairs: 0
Symbols outside eligible universe: 0


In [22]:
# Save eligible pair universe
pairDataPath = repositoryRoot/"data"/"pairDivergence"/"pairDivergencePairs.parquet"

pairUniverse.to_parquet(pairDataPath, index=False)

print(f"Saved pairs: {len(pairUniverse):,}")
print(f"Output: {pairDataPath}")

Saved pairs: 1,914,434
Output: /Users/suhas/Documents/github/Quant-Research-Lab/data/pairDivergence/pairDivergencePairs.parquet


In [23]:
# Verify saved pair universe
savedPairs = pd.read_parquet(pairDataPath)

print(f"Saved rows: {len(savedPairs):,}")
print(f"Saved columns: {savedPairs.columns.tolist()}")
print(f"Duplicate pairs: {savedPairs.duplicated().sum():,}")

Saved rows: 1,914,434
Saved columns: ['symbolA', 'symbolB']
Duplicate pairs: 0
